# 01 · Entrenamiento del modelo de detección de fraude (XGBoost)

**Caso 3 – PagoSeguro** · ITY1102 Arquitectura de Sistemas de IA · EP3

Objetivo de negocio: reducir el fraude efectivo a **< 0,1 %** y los falsos positivos a **< 5 %**, decidiendo en línea con latencia **≤ 100 ms** y con explicación SHAP de cada rechazo.

Este notebook cubre el ítem **1.1 / IE1** de la pauta (tipo de modelo, framework, proceso de entrenamiento, dataset e hiperparámetros) y deja los artefactos listos para desplegar en **AWS SageMaker** (contenedor XGBoost integrado):

| Paso | Salida |
|---|---|
| 1. Carga y validación del dataset | `data/creditcard.csv` (Kaggle mlg-ulb) |
| 2. EDA | desbalance de clases, montos, hora |
| 3. Variables | `fraud_core.build_features` (compartido con API y worker) |
| 4. Split estratificado 70/15/15 | train / validación / test |
| 5. Búsqueda de hiperparámetros | early stopping sobre PR-AUC de validación |
| 6. Selección del umbral | orientada a la meta de negocio (precisión ≥ 95 %) |
| 7. Evaluación en test | accuracy, precision, recall, F1, ROC-AUC, PR-AUC |
| 8. Latencia de inferencia | p50 / p95 / p99 por transacción |
| 9. Explicabilidad | SHAP global y local |
| 10. Exportación | `models/xgboost-model`, `model.tar.gz`, `metadata.json`, MLflow |

In [ ]:
import hashlib
import json
import os
import platform
import sys
import tarfile
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import sklearn
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

# Raíz del repo (el notebook vive en notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from fraud_core import FEATURE_COLUMNS, RAW_COLUMNS, TARGET_COLUMN, build_features

DATA_PATH = Path(os.environ.get("DATA_PATH", ROOT / "data" / "creditcard.csv"))
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

SEED = 42
TARGET_PRECISION = 0.95   # meta de negocio: < 5 % de bloqueos sobre clientes legítimos
sns.set_theme(style="whitegrid")

print(f"python {platform.python_version()} | xgboost {xgb.__version__} | "
      f"sklearn {sklearn.__version__} | shap {shap.__version__}")
print("Dataset:", DATA_PATH)

## 1. Carga y validación del dataset

Dataset: [Credit Card Fraud Detection (mlg-ulb)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284.807 transacciones de tarjetas europeas en 2 días, 492 fraudes (0,172 %). `V1..V28` son componentes PCA (los datos originales están anonimizados, lo que además es coherente con el *masking* de PAN exigido por PCI-DSS).

Se calcula el hash SHA-256 del archivo para el **lineage de datos** exigido por la auditoría trimestral.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {DATA_PATH}. Descarga el dataset desde Kaggle "
        "(mlg-ulb/creditcardfraud) y déjalo en data/creditcard.csv"
    )

def sha256_of(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

DATA_SHA256 = sha256_of(DATA_PATH)
df = pd.read_csv(DATA_PATH)

expected = RAW_COLUMNS + [TARGET_COLUMN]
assert list(df.columns) == expected, f"Esquema inesperado: {list(df.columns)}"
assert df[TARGET_COLUMN].isin([0, 1]).all()

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]} | SHA-256: {DATA_SHA256[:16]}…")
print(f"Nulos totales: {df.isna().sum().sum()} | Duplicados: {df.duplicated().sum():,}")
df.head()

## 2. Análisis exploratorio

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts().sort_index()
fraud_rate = class_counts.get(1, 0) / len(df)
print(f"Legítimas: {class_counts.get(0, 0):,} | Fraudes: {class_counts.get(1, 0):,} | "
      f"Tasa de fraude: {fraud_rate:.4%}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
class_counts.rename({0: "Legítima", 1: "Fraude"}).plot.bar(ax=axes[0], color=["#4C72B0", "#C44E52"], rot=0)
axes[0].set_yscale("log"); axes[0].set_title("Distribución de clases (escala log)")

sns.boxplot(data=df, x=TARGET_COLUMN, y="Amount", ax=axes[1], showfliers=False)
axes[1].set_xticks([0, 1], ["Legítima", "Fraude"]); axes[1].set_title("Monto por clase (sin outliers)")

hours = (df["Time"] % 86_400) // 3600
(df.assign(hour=hours).groupby("hour")[TARGET_COLUMN].mean() * 100).plot(ax=axes[2], marker="o")
axes[2].set_title("Tasa de fraude (%) por hora del día"); axes[2].set_ylabel("%")
plt.tight_layout(); plt.show()

df.groupby(TARGET_COLUMN)["Amount"].describe()

In [ ]:
# Variables PCA con mayor separación entre clases (diferencia de medias estandarizada)
pca_cols = [c for c in FEATURE_COLUMNS if c.startswith("V")]
g = df.groupby(TARGET_COLUMN)[pca_cols].mean()
sep = ((g.loc[1] - g.loc[0]) / df[pca_cols].std()).abs().sort_values(ascending=False)
top = sep.head(6).index.tolist()
print("Variables más discriminantes:", top)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), top):
    sns.kdeplot(data=df, x=col, hue=TARGET_COLUMN, common_norm=False, ax=ax, fill=True)
    ax.set_title(col)
plt.tight_layout(); plt.show()

## 3. Variables y partición de datos

`build_features` (en `fraud_core/features.py`) produce **30 variables**: `V1..V28`, `Amount` y `hour`. La misma función la usan la API FastAPI y el worker SHAP, evitando *training/serving skew*.

Partición estratificada **70 % train / 15 % validación / 15 % test**. La validación se usa para early stopping, selección de hiperparámetros y umbral; el test queda intacto hasta la evaluación final.

In [ ]:
X = build_features(df)
y = df[TARGET_COLUMN].astype(int)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

split_summary = pd.DataFrame({
    "filas": [len(X_train), len(X_val), len(X_test)],
    "fraudes": [y_train.sum(), y_val.sum(), y_test.sum()],
}, index=["train", "val", "test"])
split_summary["tasa_fraude_%"] = (split_summary.fraudes / split_summary.filas * 100).round(4)
split_summary

## 4. Entrenamiento y búsqueda de hiperparámetros

* **Modelo:** XGBoost (gradient boosted trees, `binary:logistic`) — estado del arte en datos tabulares, inferencia en < 1 ms por transacción en CPU y soportado de forma nativa por SageMaker y por `shap.TreeExplainer`.
* **Desbalance:** `scale_pos_weight = n_neg / n_pos` (sin sobremuestreo sintético, que distorsiona las probabilidades).
* **Métrica de optimización:** `aucpr` (PR-AUC), más informativa que ROC-AUC con 0,17 % de positivos.
* **Búsqueda:** muestreo aleatorio de configuraciones con early stopping (50 rondas) sobre validación.

In [ ]:
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
SCALE_POS_WEIGHT = neg / pos
print(f"scale_pos_weight = {SCALE_POS_WEIGHT:.1f}")

BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_estimators=1000,
    early_stopping_rounds=50,
    random_state=SEED,
    n_jobs=-1,
)

SEARCH_SPACE = dict(
    max_depth=[3, 4, 5, 6, 8],
    learning_rate=[0.03, 0.05, 0.1],
    subsample=[0.7, 0.8, 1.0],
    colsample_bytree=[0.6, 0.8, 1.0],
    min_child_weight=[1, 3, 5],
    gamma=[0, 0.5, 1.0],
    reg_lambda=[1, 5, 10],
    spw_factor=[1.0, 0.5, 0.25],   # fracción de scale_pos_weight (controla falsos positivos)
)
N_TRIALS = int(os.environ.get("N_TRIALS", 12))

rng = np.random.default_rng(SEED)
results = []
t0 = time.perf_counter()
for trial in range(N_TRIALS):
    cfg = {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}
    spw = SCALE_POS_WEIGHT * cfg.pop("spw_factor")
    clf = xgb.XGBClassifier(**BASE_PARAMS, **cfg, scale_pos_weight=spw)
    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    p_val = clf.predict_proba(X_val)[:, 1]
    results.append({**cfg, "scale_pos_weight": round(spw, 2),
                    "best_iteration": clf.best_iteration,
                    "val_pr_auc": average_precision_score(y_val, p_val),
                    "val_roc_auc": roc_auc_score(y_val, p_val)})
    print(f"[{trial + 1:02d}/{N_TRIALS}] PR-AUC={results[-1]['val_pr_auc']:.4f} "
          f"iter={clf.best_iteration} {cfg}")
print(f"Búsqueda completada en {time.perf_counter() - t0:.1f}s")

search_df = pd.DataFrame(results).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
search_df.head(5)

In [ ]:
# Reentrena la mejor configuración (mismo train + early stopping en validación)
best = search_df.iloc[0].to_dict()
HYPERPARAMS = {k: best[k] for k in ["max_depth", "learning_rate", "subsample", "colsample_bytree",
                                     "min_child_weight", "gamma", "reg_lambda", "scale_pos_weight"]}
HYPERPARAMS["max_depth"] = int(HYPERPARAMS["max_depth"])
HYPERPARAMS["min_child_weight"] = int(HYPERPARAMS["min_child_weight"])

model = xgb.XGBClassifier(**BASE_PARAMS, **HYPERPARAMS)
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
print("Mejores hiperparámetros:", json.dumps(HYPERPARAMS, indent=2, default=float))
print("Mejor iteración:", model.best_iteration)

hist = model.evals_result()
plt.figure(figsize=(7, 3.5))
plt.plot(hist["validation_0"]["aucpr"], label="train")
plt.plot(hist["validation_1"]["aucpr"], label="validación")
plt.axvline(model.best_iteration, ls="--", c="gray", label="early stopping")
plt.xlabel("Árboles"); plt.ylabel("PR-AUC"); plt.legend(); plt.title("Curva de aprendizaje")
plt.show()

## 5. Selección del umbral de decisión

El umbral por defecto (0,5) no es óptimo con clases tan desbalanceadas. Se elige sobre **validación**:

* **Umbral de negocio:** el que maximiza *recall* (fraude detectado) sujeto a **precisión ≥ 95 %**, es decir, menos de 5 % de los bloqueos recaen en clientes legítimos (hoy 40 %).
* Si ninguna precisión alcanza la meta, se usa el umbral que maximiza F1.

Además se define una **zona de revisión** (entre el 50 % del umbral y el umbral) para enviar a revisión manual / step-up auth en vez de rechazar.

In [ ]:
p_val = model.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, p_val)
prec, rec = prec[:-1], rec[:-1]          # alinear con thr
f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)

f1_idx = int(np.argmax(f1))
ok = np.where(prec >= TARGET_PRECISION)[0]
if len(ok):
    biz_idx = int(ok[np.argmax(rec[ok])])
    THRESHOLD, THRESHOLD_POLICY = float(thr[biz_idx]), f"max_recall@precision>={TARGET_PRECISION}"
else:
    biz_idx = f1_idx
    THRESHOLD, THRESHOLD_POLICY = float(thr[f1_idx]), "max_f1"
REVIEW_THRESHOLD = THRESHOLD / 2

print(f"Umbral F1 máx.   : {thr[f1_idx]:.4f}  (P={prec[f1_idx]:.3f}, R={rec[f1_idx]:.3f}, F1={f1[f1_idx]:.3f})")
print(f"Umbral elegido   : {THRESHOLD:.4f}  (P={prec[biz_idx]:.3f}, R={rec[biz_idx]:.3f}) política={THRESHOLD_POLICY}")
print(f"Zona de revisión : [{REVIEW_THRESHOLD:.4f}, {THRESHOLD:.4f})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thr, prec, label="precision"); ax.plot(thr, rec, label="recall"); ax.plot(thr, f1, label="F1")
ax.axvline(THRESHOLD, c="k", ls="--", label=f"umbral={THRESHOLD:.3f}")
ax.axhline(TARGET_PRECISION, c="gray", ls=":", label="meta precisión")
ax.set_xscale("log"); ax.set_xlabel("umbral (log)"); ax.legend(); ax.set_title("Métricas vs umbral (validación)")
plt.show()

## 6. Evaluación final en test (datos nunca vistos)

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
y_pred = (p_test >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
METRICS = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, p_test),
    "pr_auc": average_precision_score(y_test, p_test),
    # Métricas de negocio
    "false_positive_share_of_blocks": fp / max(tp + fp, 1),   # meta < 5 %
    "false_positive_rate": fp / max(fp + tn, 1),
    "missed_fraud_rate": fn / max(fn + tp, 1),
}
amount_test = X_test["Amount"].to_numpy()
fraud_amount = amount_test[y_test.to_numpy() == 1].sum()
missed_amount = amount_test[(y_test.to_numpy() == 1) & (y_pred == 0)].sum()
METRICS["fraud_amount_caught_%"] = 1 - missed_amount / max(fraud_amount, 1e-9)
METRICS["residual_fraud_loss_%_of_volume"] = missed_amount / amount_test.sum()  # meta < 0,1 %

print(classification_report(y_test, y_pred, target_names=["Legítima", "Fraude"], digits=4))
pd.Series(METRICS, name="test").to_frame().style.format("{:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Legítima", "Fraude"],
                                        cmap="Blues", values_format=",d", ax=axes[0])
axes[0].set_title(f"Matriz de confusión (umbral={THRESHOLD:.3f})")
pt, rt, _ = precision_recall_curve(y_test, p_test)
axes[1].plot(rt, pt); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"Curva Precision-Recall (PR-AUC={METRICS['pr_auc']:.3f})")
plt.tight_layout(); plt.show()

objectives = pd.DataFrame({
    "meta": ["< 5 %", "< 0,1 %"],
    "resultado": [f"{METRICS['false_positive_share_of_blocks']:.2%}",
                  f"{METRICS['residual_fraud_loss_%_of_volume']:.3%}"],
    "cumple": [METRICS["false_positive_share_of_blocks"] < 0.05,
               METRICS["residual_fraud_loss_%_of_volume"] < 0.001],
}, index=["Falsos positivos (sobre bloqueos)", "Pérdida por fraude (sobre volumen $)"])
objectives

## 7. Latencia de inferencia

Requisito del caso: **≤ 100 ms** extremo a extremo (P99 < 90 ms). Se mide el costo del modelo para una transacción individual (como llega en el flujo de pago) usando el `Booster` nativo, que es lo que ejecuta el contenedor de SageMaker.

In [ ]:
booster = model.get_booster()
booster.set_param({"nthread": 1})
sample = X_test.iloc[:1]
dsample = xgb.DMatrix(sample, feature_names=FEATURE_COLUMNS)
for _ in range(50):
    booster.predict(dsample, iteration_range=(0, model.best_iteration + 1))  # warm-up

lat_ms = []
for i in range(1000):
    row = X_test.iloc[[i % len(X_test)]]
    t = time.perf_counter()
    booster.predict(xgb.DMatrix(row, feature_names=FEATURE_COLUMNS),
                    iteration_range=(0, model.best_iteration + 1))
    lat_ms.append((time.perf_counter() - t) * 1000)

LATENCY = {f"p{q}_ms": float(np.percentile(lat_ms, q)) for q in (50, 95, 99)}
print({k: round(v, 3) for k, v in LATENCY.items()})

## 8. Explicabilidad (SHAP)

`shap.TreeExplainer` calcula valores SHAP exactos para modelos de árboles. En producción esto lo hace el **worker asíncrono** (consume de RabbitMQ) para no sumar latencia al flujo de pago, y guarda el reporte de auditoría de cada rechazo (requisito CMF).

In [ ]:
explainer = shap.TreeExplainer(model)
X_shap = X_test.sample(min(2000, len(X_test)), random_state=SEED)
X_shap = pd.concat([X_shap, X_test[y_test == 1]]).drop_duplicates()
shap_values = explainer.shap_values(X_shap)

shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
plt.title("Importancia global SHAP"); plt.tight_layout(); plt.show()

GLOBAL_IMPORTANCE = (pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURE_COLUMNS)
                     .sort_values(ascending=False))
GLOBAL_IMPORTANCE.head(10).round(4)

In [ ]:
# Explicación local de una transacción rechazada: el reporte que recibe el auditor
rejected = X_test[(p_test >= THRESHOLD)]
idx = rejected.index[0] if len(rejected) else X_test.index[int(np.argmax(p_test))]
x_one = X_test.loc[[idx]]
sv_one = explainer(x_one)

print(f"Transacción {idx} | prob. fraude = {model.predict_proba(x_one)[0, 1]:.4f} | "
      f"real = {'Fraude' if y_test.loc[idx] == 1 else 'Legítima'}")
shap.plots.waterfall(sv_one[0], max_display=10, show=False)
plt.tight_layout(); plt.show()

top_reasons = (pd.Series(sv_one.values[0], index=FEATURE_COLUMNS)
               .sort_values(key=np.abs, ascending=False).head(5))
top_reasons.to_frame("shap").assign(valor=x_one.iloc[0][top_reasons.index])

## 9. Exportación de artefactos

| Archivo | Uso |
|---|---|
| `models/xgboost-model` | Booster nativo (JSON). Nombre requerido por el contenedor XGBoost integrado de SageMaker |
| `models/model.tar.gz` | Paquete que se sube a S3 para crear el `Model` / `Endpoint` de SageMaker |
| `models/metadata.json` | Orden de variables, umbrales, métricas, hiperparámetros, versiones y hash del dataset (lineage) |
| `models/sample_payloads.json` | Transacciones reales (legítima y fraude) para tests de la API y Locust |

Si `mlflow` está instalado, se registra además el run (parámetros, métricas, modelo y metadata) en `MLFLOW_TRACKING_URI` (por defecto `./mlruns`).

In [ ]:
# Booster con exactamente los árboles hasta best_iteration (lo que ve SageMaker)
final_booster = model.get_booster()[: model.best_iteration + 1]
model_path = MODEL_DIR / "xgboost-model"
final_booster.save_model(model_path.with_suffix(".json"))   # formato JSON
model_path.with_suffix(".json").replace(model_path)          # nombre exigido por SageMaker

with tarfile.open(MODEL_DIR / "model.tar.gz", "w:gz") as tar:
    tar.add(model_path, arcname="xgboost-model")

# Sanity check: el booster exportado reproduce las probabilidades
reloaded = xgb.Booster(); reloaded.load_model(model_path)
np.testing.assert_allclose(reloaded.predict(xgb.DMatrix(X_test, feature_names=FEATURE_COLUMNS)),
                           p_test, rtol=1e-5)

trained_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
MODEL_VERSION = f"xgb-{trained_at[:10].replace('-', '')}-{DATA_SHA256[:8]}"
metadata = {
    "model_name": "pagoseguro-fraud-xgboost",
    "model_version": MODEL_VERSION,
    "trained_at": trained_at,
    "framework": {"xgboost": xgb.__version__, "sklearn": sklearn.__version__, "python": platform.python_version()},
    "sagemaker": {"container": "xgboost", "container_version": "3.0-5", "content_type": "text/csv"},
    "feature_columns": FEATURE_COLUMNS,
    "threshold": THRESHOLD,
    "review_threshold": REVIEW_THRESHOLD,
    "threshold_policy": THRESHOLD_POLICY,
    "hyperparameters": {**{k: float(v) for k, v in HYPERPARAMS.items()},
                        "n_estimators": int(model.best_iteration + 1)},
    "metrics_test": {k: float(v) for k, v in METRICS.items()},
    "latency_model_only": LATENCY,
    "global_shap_importance_top10": GLOBAL_IMPORTANCE.head(10).round(6).to_dict(),
    "data_lineage": {
        "source": "kaggle:mlg-ulb/creditcardfraud",
        "file": DATA_PATH.name,
        "sha256": DATA_SHA256,
        "rows": int(len(df)),
        "fraud_rate": float(fraud_rate),
        "split": {"train": len(X_train), "val": len(X_val), "test": len(X_test),
                  "strategy": "stratified 70/15/15", "seed": SEED},
    },
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

raw_test = df.loc[X_test.index, RAW_COLUMNS]
samples = {
    "legit": raw_test[y_test == 0].head(3).to_dict(orient="records"),
    "fraud": raw_test[y_test == 1].head(3).to_dict(orient="records"),
}
(MODEL_DIR / "sample_payloads.json").write_text(json.dumps(samples, indent=2))

for p in sorted(MODEL_DIR.iterdir()):
    print(f"{p.name:24s} {p.stat().st_size / 1024:8.1f} KB")
print("Versión:", MODEL_VERSION)

In [ ]:
# Registro en MLflow (versionado estricto exigido por la auditoría trimestral)
try:
    import mlflow
except ImportError:
    print("mlflow no está instalado: se omite el registro (pip install mlflow)")
else:
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", f"file:{ROOT / 'mlruns'}"))
    mlflow.set_experiment("pagoseguro-fraud")
    with mlflow.start_run(run_name=MODEL_VERSION):
        mlflow.log_params({**HYPERPARAMS, "threshold": THRESHOLD, "data_sha256": DATA_SHA256})
        mlflow.log_metrics({k.replace("%", "pct"): float(v) for k, v in {**METRICS, **LATENCY}.items()})
        mlflow.log_artifact(str(MODEL_DIR / "metadata.json"))
        mlflow.xgboost.log_model(final_booster, name="model")
    print("Run registrado en MLflow:", mlflow.get_tracking_uri())

## 10. Conclusiones y siguientes pasos

* El modelo y su umbral quedan versionados en `models/metadata.json` junto al hash del dataset.
* **Siguiente:** `sagemaker/deploy.py` sube `model.tar.gz` a S3 y crea el endpoint (contenedor XGBoost integrado, `ml.c5.large`/`ml.m5.large`, auto-scaling por `InvocationsPerInstance`).
* La API FastAPI usa `fraud_core.build_features` → invoca el endpoint → aplica `threshold`/`review_threshold` → publica el evento en RabbitMQ para que el worker calcule SHAP y persista el reporte de auditoría.